# Лабораторна робота 4 — FP32 проти INT8

Ноутбук показує той самий експеримент, який реалізований у `lab4.py`. Щоб запустити повний експорт, квантизацію й заміри, встановіть `RUN_EXPERIMENT = True` у наступній клітинці. Дані й checkpoint беруться з `../data/` та `../lab3/`.

Для ваг згортки діапазон каналу `α_c = max |W_c|`, а втрата точності типового каналу при спільному масштабі оцінюється як `log₂(max α_c / median α_c)` бітів.

In [ ]:
from pathlib import Path
import subprocess
import sys

RUN_EXPERIMENT = False
if RUN_EXPERIMENT:
    subprocess.run([
        sys.executable, 'lab4.py',
        '--model', '../lab3/lab3_model.pt',
        '--split', '../lab3/split.csv',
        '--data', '../data/intel-image-classification',
        '--out', '.',
    ], check=True)

## Парне порівняння якості

Для `n = 1502`, `b` означає INT8 правильно та FP32 помилився, `c` — навпаки. Різниця точності `Δacc = (b-c)/n`, її парна похибка `SE ≈ √(b+c)/n`. Якщо `|b-c| ≤ 2√(b+c)`, зміна accuracy не відрізняється від шуму.

До запуску очікується, що per-tensor знизить згоду прогнозів із FP32, а зміна accuracy лишиться в межах цього шумового критерію.

In [ ]:
import json
from IPython.display import display

result_path = Path('lab4_results.json')
if not result_path.exists():
    raise FileNotFoundError('Спочатку виконайте команду з README або встановіть RUN_EXPERIMENT = True.')
result = json.loads(result_path.read_text(encoding='utf-8'))
q = result['quality']
display([
    {'Варіант': 'PyTorch FP32', **q['pytorch_fp32']},
    {'Варіант': 'ONNX FP32', **q['onnx_fp32'], 'згода з FP32': 1.0},
    {'Варіант': 'INT8 per-channel', **q['int8_per_channel']},
    {'Варіант': 'INT8 per-tensor', **q['int8_per_tensor']},
])
display([
    {'Варіант': 'FP32', **result['latency']['fp32']},
    {'Варіант': 'INT8', **result['latency']['int8']},
])
display(result['export'])
display(result['quantization'])
display({'per_tensor_seconds': result['per_tensor_experiment_seconds']})
display(result['weight_ranges'])
display(result['environment'])

## Результат

Числа беруться з JSON, створеного скриптом. p50/p95 обчислено за 300 викликами `session.run`; критерій зміни затримки використовує неперетин діапазонів раундових p50. Цей запуск відбувся від батареї через відключення електрики; середовище записане у звіті. Після відновлення живлення запустіть команду README без `--allow-battery`, щоб оновити затримки. GPT використовувався для підготовки й перевірки коду; обчислені значення отримані локальним запуском.